In [6]:
# Importing statements
import numpy as np
import tensorflow as tf
import scipy
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib as plt
import keras
import json
# import shap


#Import and organise files

In [3]:
# title_akas = pd.read_csv('E:\\University Stuff\\Third year\\Dissertation\\Datasets\\title_akas.tsv', sep='\t')
# title_basics = pd.read_csv("E:\\University Stuff\\Third year\\Dissertation\\Datasets\\title_basics.tsv", sep='\t')
# title_crew = pd.read_csv("E:\\University Stuff\\Third year\\Dissertation\\Datasets\\title_crew.tsv", sep='\t')
# # title_episode = pd.read_csv("E:\\University Stuff\\Third year\\Dissertation\\Datasets\\title_episode.tsv", sep='\t')
# title_principals = pd.read_csv("E:\\University Stuff\\Third year\\Dissertation\\Datasets\\title_principals.tsv", sep='\t')
# title_ratings = pd.read_csv('E:\\University Stuff\\Third year\\Dissertation\\Datasets\\title_ratings.tsv', sep='\t')
# # name_basics = pd.read_csv("E:\\University Stuff\\Third year\\Dissertation\\Datasets\\name_basics.tsv", sep='\t')



Using Kaggle dataset for ratings

In [4]:
kaggle_ratings = pd.read_csv('E:\\University Stuff\\Third year\\Dissertation\\kaggle_dataset\\ratings.csv')
kaggle_metadata = pd.read_csv('E:\\University Stuff\\Third year\\Dissertation\\kaggle_dataset\\movies_metadata.csv')
print(len(kaggle_metadata))

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\University Stuff\\Third year\\Dissertation\\kaggle_dataset\\ratings.csv'

In [ ]:
movie_metadata = kaggle_metadata.drop(columns=['belongs_to_collection', 'homepage', 'status',
                                               'title', 'video', 'original_title', 'overview',
                                               'production_companies', 'production_countries',
                                               'spoken_languages', 'tagline', 'poster_path',
                                               'id', 'imdb_id', 'genres', 'original_language', 'adult'])
movie_metadata = movie_metadata.dropna()
print(len(movie_metadata))
movie_metadata.head()

In [ ]:

# Converting release year from string to int
def year_to_int(s):
    if type(s) == str:
        return float(s[:4])
    else:
        return pd.NA

def str_to_int(s):
    try:
        return float(s)
    except:
        print(s)
        return 'flagged'

def delete_broken_entries(series):
    for ind, row in series.iterrows():
        if row['budget'] == 'flagged':
            # print(ind)
            series = series.drop(index= ind)

    return series

movie_metadata['release_date'] = movie_metadata['release_date'].apply(year_to_int)

# Cleaning broken entries
movie_metadata['budget'] = movie_metadata['budget'].apply(str_to_int)
movie_metadata = delete_broken_entries(movie_metadata)


In [ ]:
budget_sum = 0
revenue_sum = 0
for ind, row in movie_metadata.iterrows():
    if row['budget'] != 0:
        budget_sum += row['budget']
    if row['revenue'] != 0:
        revenue_sum += row['revenue']


In [ ]:
labels = movie_metadata['vote_average']
movie_metadata = movie_metadata.drop(columns=['vote_average'])

Visualising Data

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(movie_metadata, labels, test_size=0.33)

x_train = np.asarray(x_train.fillna(value=0.0)).astype(np.float64)
x_test = np.asarray(x_test.fillna(value=0.0)).astype(np.float64)
np.multiply(y_train, 10)
np.multiply(y_test, 10)

y_train = np.asarray(y_train).astype(int)
y_test = np.asarray(y_test).astype(int)


Developing Neural Network

In [ ]:
movie_metadata.query("budget == '/ff9qCepilowshEtG2GYWwzt2bs4.jpg' or budget == '/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg' or budget == '/zaSf5OG7V8X8gqFvly88zDdRm46.jpg' ")
# x_train.query("release_date == 'NAType'")
# x_train.query("revenue == 'NAType'")
# x_train.query("runtime == 'NAType'")
# x_train.query("vote_count == 'NAType'")

In [ ]:
# Shaping labels
print(x_train.shape, ' ', y_train.shape, ' X_train and Y_train ')
print(x_test.shape, ' ', y_test.shape, ' X_test and Y_test')

In [ ]:
from keras.layers import Dense
from keras import Sequential

# create model
model = Sequential()
model.add(Dense(24, input_dim=6, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(101, activation='softmax'))

# Compile model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x = x_train, y= y_train, verbose=1, epochs=30, shuffle = True, validation_data=(x_test, y_test))

In [ ]:
score = model.evaluate(x_test, y_test, verbose=1)
print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')

# Visualize history
# Plot history: Loss
plt.plot(history.history['val_loss'])
plt.title('Validation loss history')
plt.ylabel('Loss value')
plt.xlabel('No. epoch')
plt.show()

# Plot history: Accuracy
plt.plot(history.history['val_accuracy'])
plt.title('Validation accuracy history')
plt.ylabel('Accuracy value (%)')
plt.xlabel('No. epoch')
plt.show()

In [ ]:
predictions = model.predict(x_test, 20, verbose= 1)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, predictions.argmax(axis=1))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(set(y_test)),)
fig, ax = plt.subplots(figsize=(20,20))
ax.set_title('DNN confusion matrix')
disp.plot(ax=ax, cmap=plt.cm.Blues)
fig.autofmt_xdate()

In [ ]:
# creating the dataset
data = {'C':20, 'C++':15, 'Java':30,
        'Python':35}
courses = list(data.keys())
values = list(data.values())

fig = plt.figure(figsize = (20, 20))

# creating the bar plot
# plt.scatter(movie_metadata['release_date'], list(labels), color ='blue', width = 0.4)
plt.scatter(movie_metadata['release_date'], list(labels), c='blue')

plt.xlabel("Release Year")
plt.ylabel("Rating")
plt.title("Kaggle Data")
plt.show()

In [ ]:
explainer = shap.KernelExplainer(model.predict, x_train[30200:])
shap_values = explainer.shap_values(x_test,nsamples=100)

In [ ]:
features = ['budget','popularity','release_date','revenue','runtime','vote_count']
shap.summary_plot(shap_values,x_test, feature_names= features)

In [ ]:
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0,:]  ,x_test[0,:],feature_names=features)